In [1]:
import subprocess as sp

In [2]:
from scintkit.pipelines.auto import process

outputs = process(
    "mearem_lvl0.pq",
    verbose=True,
    mode="both"
)

print(outputs)

Processing mearem_lvl0.pq...
Reading and formatting parquet file: mearem_lvl0.pq...
Error processing mearem_lvl0.pq
[Errno 2] No such file or directory: 'mearem_lvl0.pq'
[]


In [3]:
'''import pandas as pd

df = pd.read_parquet("/Users/dal674840/Downloads/scintpi3_20241011_0004_96.7572W_32.9920N_v326f_lvl0.pq")
df2 = pd.read_parquet("/Users/dal674840/Downloads/cg01001a00.26__mearem.parquet")

print("=== df2 (first 50 rows) ===")
print(df2.head(50))

print("\n=== df (first 50 rows) ===")
print(df.head(50))

print("df2 columns:")
print(df2.columns.tolist())

print("\ndf columns:")
print(df.columns.tolist())'''

import pandas as pd
import numpy as np

# --------------------------------------------------
# Read Septentrio parquet
# --------------------------------------------------

df = pd.read_parquet(
    "/Users/dal674840/Downloads/cg01001a00.26__mearem.parquet"
)

# datetime is currently the index
df = df.reset_index()

# Ensure datetime column exists
df["datetime"] = pd.to_datetime(df["datetime"])

# --------------------------------------------------
# Signal mapping
# --------------------------------------------------

mapping = {

    # Sig1
    'GPS_L1CA': (1575.42, 'Sig1'),
    'GLO_L1CA': (1602.00, 'Sig1'),
    'GEO_L1': (1575.42, 'Sig1'),
    'QZS_L1CA': (1575.42, 'Sig1'),
    'GAL_L1BC': (1575.42, 'Sig1'),
    'GAL_E1': (1575.42, 'Sig1'),
    'GAL_E1BC': (1575.42, 'Sig1'),
    'BDS_B1I': (1561.098, 'Sig1'),
    'IRNSS_L5': (1176.45, 'Sig1'),

    # Sig2
    'GPS_L2C': (1227.60, 'Sig2'),
    'GLO_L2C': (1246.00, 'Sig2'),
    'GLO_L2CA': (1246.60, 'Sig2'),
    'QZS_L2C': (1227.60, 'Sig2'),
    'GAL_E5a': (1176.45, 'Sig2'),
    'SBAS_L5': (1176.45, 'Sig2'),
    'BDS_B2I': (1207.14, 'Sig2'),
    'GEO_L5': (1176.45, 'Sig2'),

    # Sig3
    'GPS_L5': (1176.45, 'Sig3'),
    'QZS_L5': (1176.45, 'Sig3'),
    'GAL_E5b': (1207.14, 'Sig3'),
    'BDS_B3I': (1268.52, 'Sig3'),

    # Sig4
    'GPS_L2PY': (1227.60, 'Sig4'),
    'GPS_L1P': (1575.42, 'Sig4'),
    'GLO_L1P': (1602.00, 'Sig4'),
    'GLO_L2P': (1246.00, 'Sig4'),
    'GAL_E5': (1191.795, 'Sig4'),
    'GAL_E6BC': (1278.75, 'Sig4'),
    'GLO_L3': (1202.025, 'Sig4')
}

# --------------------------------------------------
# Assign frequencies and signal groups
# --------------------------------------------------

df["freq"] = df["SIG"].map(
    lambda x: mapping.get(x, (np.nan, None))[0]
)

df["sig_group"] = df["SIG"].map(
    lambda x: mapping.get(x, (np.nan, None))[1]
)

# Keep only mapped signals
df = df[df["sig_group"].notna()].copy()

# --------------------------------------------------
# Signal names
# --------------------------------------------------

sig_pivot = (
    df.pivot_table(
        index=["datetime", "SVID"],
        columns="sig_group",
        values="SIG",
        aggfunc="first"
    )
    .rename(columns={
        "Sig1": "sig_1",
        "Sig2": "sig_2",
        "Sig3": "sig_3",
        "Sig4": "sig_4"
    })
)

# --------------------------------------------------
# Frequencies
# --------------------------------------------------

freq_pivot = (
    df.pivot_table(
        index=["datetime", "SVID"],
        columns="sig_group",
        values="freq",
        aggfunc="first"
    )
    .rename(columns={
        "Sig1": "freq_1",
        "Sig2": "freq_2",
        "Sig3": "freq_3",
        "Sig4": "freq_4"
    })
)

# --------------------------------------------------
# SNR
# --------------------------------------------------

snr_pivot = (
    df.pivot_table(
        index=["datetime", "SVID"],
        columns="sig_group",
        values="SNR",
        aggfunc="first"
    )
    .rename(columns={
        "Sig1": "snr1",
        "Sig2": "snr2",
        "Sig3": "snr3",
        "Sig4": "snr4"
    })
)

# --------------------------------------------------
# Carrier phase
# --------------------------------------------------

phase_pivot = (
    df.pivot_table(
        index=["datetime", "SVID"],
        columns="sig_group",
        values="Phase",
        aggfunc="first"
    )
    .rename(columns={
        "Sig1": "cph1",
        "Sig2": "cph2",
        "Sig3": "cph3",
        "Sig4": "cph4"
    })
)

# --------------------------------------------------
# Pseudorange
# --------------------------------------------------

pr_pivot = (
    df.pivot_table(
        index=["datetime", "SVID"],
        columns="sig_group",
        values="PR",
        aggfunc="first"
    )
    .rename(columns={
        "Sig1": "rng1",
        "Sig2": "rng2",
        "Sig3": "rng3",
        "Sig4": "rng4"
    })
)

# --------------------------------------------------
# Merge everything
# --------------------------------------------------

lvl0 = pd.concat(
    [
        sig_pivot,
        freq_pivot,
        snr_pivot,
        phase_pivot,
        pr_pivot
    ],
    axis=1
).reset_index()

# --------------------------------------------------
# Satellite information
# --------------------------------------------------

constellation_map = {
    'G': 'GPS',
    'R': 'GLO',
    'E': 'GAL',
    'C': 'BDS',
    'J': 'QZS',
    'I': 'IRNSS',
    'S': 'SBAS'
}

lvl0["prn"] = lvl0["SVID"]

lvl0["cons"] = (
    lvl0["prn"]
    .str[0]
    .map(constellation_map)
)

lvl0["svid"] = (
    lvl0["prn"]
    .str[1:]
    .astype(int)
)

# --------------------------------------------------
# Extra columns
# --------------------------------------------------

lvl0["minbin"] = lvl0["datetime"]

lvl0["elev"] = 45.0
lvl0["azim"] = 45.0

# Create explicit index column
lvl0.insert(
    0,
    "index",
    np.arange(len(lvl0))
)

# --------------------------------------------------
# Exact column order
# --------------------------------------------------

desired_cols = [
    'index',
    'cons',
    'svid',
    'elev',
    'azim',
    'snr1',
    'snr2',
    'cph1',
    'cph2',
    'rng1',
    'rng2',
    'datetime',
    'minbin',
    'prn',
    'sig_1',
    'sig_2',
    'sig_3',
    'freq_1',
    'freq_2',
    'freq_3'
]

# Create missing columns if needed
for col in desired_cols:
    if col not in lvl0.columns:
        lvl0[col] = np.nan

lvl0 = lvl0[desired_cols]

# --------------------------------------------------
# Sort
# --------------------------------------------------

lvl0 = lvl0.sort_values(
    ["datetime", "prn"]
).reset_index(drop=True)

# --------------------------------------------------
# Save
# --------------------------------------------------

lvl0.to_parquet(
    "mearem_lvl0.pq",
    index=False
)

# --------------------------------------------------
# Verify
# --------------------------------------------------

print("\nColumns:")
print(lvl0.columns.tolist())

print("\nShape:")
print(lvl0.shape)

print("\nFirst 5 rows:")
print(lvl0.head())


Columns:
['index', 'cons', 'svid', 'elev', 'azim', 'snr1', 'snr2', 'cph1', 'cph2', 'rng1', 'rng2', 'datetime', 'minbin', 'prn', 'sig_1', 'sig_2', 'sig_3', 'freq_1', 'freq_2', 'freq_3']

Shape:
(902464, 20)

First 5 rows:
sig_group  index cons  svid  elev  azim     snr1     snr2          cph1  \
0              0  BDS    20  45.0  45.0  49.6250      NaN  1.250981e+08   
1              1  BDS    32  45.0  45.0  51.9375      NaN  1.134868e+08   
2              2  BDS    37  45.0  45.0  45.3750      NaN  1.224824e+08   
3              3  GAL     2  45.0  45.0  46.0625  46.1875  1.305463e+08   
4              4  GAL     3  45.0  45.0  45.0000  45.9375  1.303613e+08   

sig_group          cph2          rng1          rng2   datetime     minbin  \
0                   NaN  2.402377e+07           NaN 2026-01-01 2026-01-01   
1                   NaN  2.179393e+07           NaN 2026-01-01 2026-01-01   
2                   NaN  2.352144e+07           NaN 2026-01-01 2026-01-01   
3          9.748594

In [4]:
from scintkit.pipelines.auto import process

outputs = process(
    "mearem_lvl0.pq",
    verbose=True,
    mode="both"
)

print(outputs)

Processing mearem_lvl0.pq...
Reading and formatting parquet file: mearem_lvl0.pq...
Ensuring format...
Processing phases...
Computing TEC...
many cycle slips detected for SVID E02, 45000/45000.
many cycle slips detected for SVID E03, 45000/45000.
many cycle slips detected for SVID E08, 45000/45000.
many cycle slips detected for SVID E18, 45000/45000.
many cycle slips detected for SVID E30, 45000/45000.


/Users/dal674840/scintkit/src/scintkit/services/compute.py:64: RuntimeWarning: Mean of empty slice
  carrier = carrier - np.nanmean(carrier)
/Users/dal674840/scintkit/src/scintkit/services/compute.py:64: RuntimeWarning: Mean of empty slice
  carrier = carrier - np.nanmean(carrier)
/Users/dal674840/scintkit/src/scintkit/services/compute.py:64: RuntimeWarning: Mean of empty slice
  carrier = carrier - np.nanmean(carrier)
/Users/dal674840/scintkit/src/scintkit/services/compute.py:64: RuntimeWarning: Mean of empty slice
  carrier = carrier - np.nanmean(carrier)
/Users/dal674840/scintkit/src/scintkit/services/compute.py:64: RuntimeWarning: Mean of empty slice
  carrier = carrier - np.nanmean(carrier)
/Users/dal674840/scintkit/src/scintkit/services/compute.py:64: RuntimeWarning: Mean of empty slice
  carrier = carrier - np.nanmean(carrier)


many cycle slips detected for SVID S23, 10579/45000.
Computing products...
Merging products back to original dataframe...
Finished processing mearem_lvl0.pq.
['mearem_lvl3.pq', 'mearem_lvl2.pq']


In [ ]:
import pandas as pd


lvl3 = pd.read_parquet("/Users/dal674840/Downloads/scintpi3_20241011_0004_96.7572W_32.9920N_v326f_lvl0.pq")


print("\nLVL3")
print(lvl3.columns.tolist())
print(lvl3.head())

LVL2
['secbin', 'prn', 'level_0', 'index', 'cons', 'svid', 'elev', 'azim', 'snr1', 'snr2', 'cph1', 'cph2', 'rng1', 'rng2', 'datetime', 'minbin', 'sig_1', 'sig_2', 'sig_3', 'freq_1', 'freq_2', 'freq_3', 'detrended_cph1', 'cycleslips_cph1', 'edgegap_mask_cph1', 'detrended_cph2', 'cycleslips_cph2', 'edgegap_mask_cph2', 'v1', 'v2', 'clock_term', 'detrended_noclk_cph1', 'detrended_noclk_cph2', 'tec_cph12', 'tec_rng12', 'sigma_phi_1', 'n_1', 'n_cycleslip_1', 'quality_1', 's4_1', 's4_corrected_1', 'sigma_phi_2', 'n_2', 'n_cycleslip_2', 'quality_2', 's4_2', 's4_corrected_2']
      secbin  prn  level_0  index cons  svid  elev  azim     snr1     snr2  \
0 2026-01-01  C20        0      0  BDS    20  45.0  45.0  49.6250      NaN   
1 2026-01-01  C32        1      1  BDS    32  45.0  45.0  51.9375      NaN   
2 2026-01-01  C37        2      2  BDS    37  45.0  45.0  45.3750      NaN   
3 2026-01-01  E02        3      3  GAL     2  45.0  45.0  46.0625  46.1875   
4 2026-01-01  E03        4      4  G

In [6]:
file = '/Users/dal674840/Downloads/scintpi3_20241011_0004_96.7572W_32.9920N_v326f_lvl0.pq'

import pandas as pd

lvl2 = pd.read_parquet("/Users/dal674840/Downloads/scintpi3_20241011_0004_96.7572W_32.9920N_v326f_lvl2.pq")
lvl3 = pd.read_parquet("/Users/dal674840/Downloads/scintpi3_20241011_0004_96.7572W_32.9920N_v326f_lvl3.pq")

print("LVL2")
print(lvl2.columns.tolist())
print(lvl2.head())

print("\nLVL3")
print(lvl3.columns.tolist())
print(lvl3.head())


LVL2
['secbin', 'prn', 'level_0', 'index', 'cons', 'svid', 'elev', 'azim', 'snr1', 'snr2', 'cph1', 'cph2', 'rng1', 'rng2', 'datetime', 'minbin', 'sig_1', 'sig_2', 'sig_3', 'freq_1', 'freq_2', 'freq_3', 'detrended_cph1', 'cycleslips_cph1', 'edgegap_mask_cph1', 'detrended_cph2', 'cycleslips_cph2', 'edgegap_mask_cph2', 'v1', 'v2', 'clock_term', 'detrended_noclk_cph1', 'detrended_noclk_cph2', 'tec_cph12', 'tec_rng12', 'sigma_phi_1', 'n_1', 'n_cycleslip_1', 'quality_1', 's4_1', 's4_corrected_1', 'sigma_phi_2', 'n_2', 'n_cycleslip_2', 'quality_2', 's4_2', 's4_corrected_2']
               secbin  prn  level_0  index cons  svid  elev  azim  snr1  snr2  \
0 2024-10-11 00:05:06  C11        6      6  BDS    11    14    51  39.0  45.0   
1 2024-10-11 00:05:06  C20        4      4  BDS    20    42   282  51.0   NaN   
2 2024-10-11 00:05:06  C23        5      5  BDS    23    41    55  48.0   NaN   
3 2024-10-11 00:05:06  C28        7      7  BDS    28    25   139  46.0   NaN   
4 2024-10-11 00:05:06